In [ ]:
import os
import random
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load data
# ────────────────────────────────────────────────
data = pd.read_csv("/Users/othmanbensouda/Desktop/debt_collection_website/files/get_plaintiffs_defendants.csv")

# ────────────────────────────────────────────────
# 3. Assign annotators randomly (Brian, Parker, Victor)
# ────────────────────────────────────────────────
cases = list(data["CASE_NUMBER"].unique())
random.seed(42)
random.shuffle(cases)
n = len(cases)

part1, part2, part3 = cases[: n // 3], cases[n // 3 : 2 * n // 3], cases[2 * n // 3 :]

def assign_annotator(case_number):
    if case_number in part1:
        return "Brian"
    elif case_number in part2:
        return "Parker"
    else:
        return "Victor"

data["annotator_id"] = data["CASE_NUMBER"].apply(assign_annotator)

# ────────────────────────────────────────────────
# 4. Clean up text fields (avoid NaN/float errors)
# ────────────────────────────────────────────────
data["FULL_NAME"] = data["FULL_NAME"].fillna("").astype(str)
data["PERSON_ROLE"] = data["PERSON_ROLE"].fillna("").astype(str)

# ────────────────────────────────────────────────
# 5. Aggregate one row per case
# ────────────────────────────────────────────────
grouped = (
    data.groupby("CASE_NUMBER")
    .apply(lambda g: pd.Series({
        "case_number": g.name,
        "case_hashkey": g["CASE_HASHKEY"].iloc[0] if "CASE_HASHKEY" in g else None,
        "complaint_filed_date": g["COMPLAINT_FILED_DATE"].iloc[0] if "COMPLAINT_FILED_DATE" in g else None,
        "plaintiff": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "plaintiff", "FULL_NAME"] if pd.notna(x) and x],
        "defendant": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "defendant", "FULL_NAME"] if pd.notna(x) and x],
        "annotator_id": g["annotator_id"].iloc[0],
    }))
    .reset_index(drop=True)
)

# ────────────────────────────────────────────────
# 6. Upload to Supabase (batch, fast)
# ────────────────────────────────────────────────
records = grouped.to_dict(orient="records")

response = supabase.table("cases_gold").upsert(records).execute()

print("✅ Upload complete!")
print("Inserted/updated rows:", len(records))
print("\nAnnotator distribution:")
print(grouped["annotator_id"].value_counts())

# Optional: preview one example
print("\nExample row:")
print(grouped.head(1).to_dict(orient="records")[0])


/var/folders/k7/b0_b7t6j6n72t68sh4s7t8400000gn/T/ipykernel_28487/84020101.py:51: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


✅ Upload complete!
Inserted/updated rows: 5

Annotator distribution:
annotator_id
Victor    2
Parker    2
Brian     1
Name: count, dtype: int64

Example row:
{'case_number': '23CHLC16737', 'case_hashkey': 221885112732742272, 'complaint_filed_date': '2023-06-29 00:00:00.000', 'plaintiff': ['DNF Associates, LLC'], 'defendant': ['SPENCY ARINGO'], 'annotator_id': 'Victor'}


In [ ]:
import openpyxl
grouped.to_excel("/Users/othmanbensouda/Desktop/debt_collection_website/files/cases_assigned.xlsx")

# Round 2 (inter-annotator agreement)

In [3]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ───────────────────────────────────────────
# 1. Load Supabase credentials
# ───────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ───────────────────────────────────────────
# 2. Fetch ROUND 1 cases
# ───────────────────────────────────────────
res = (
    supabase.table("cases_gold")
    .select("*")
    .eq("round", 1)
    .execute()
)

raw_df = pd.DataFrame(res.data)
print("Loaded round 1 cases total:", len(raw_df))

# Keep ONLY the 3 annotators involved in IAA
df = raw_df[raw_df["annotator_id"].isin(["Parker", "Brian", "Victor"])].copy()

print("Cases retained for IAA:", len(df))
print(df["annotator_id"].value_counts())

# Remove accidental duplicates
df = df.drop_duplicates(subset=["case_number"])

# ───────────────────────────────────────────
# 3. Stable perfect split (deterministic)
# ───────────────────────────────────────────
def split_half(lst):
    lst = sorted(lst)
    n = len(lst)
    half = n // 2
    return lst[:half], lst[half:]

# Split by annotator
parker_cases = df[df.annotator_id == "Parker"]["case_number"].tolist()
brian_cases  = df[df.annotator_id == "Brian"]["case_number"].tolist()
victor_cases = df[df.annotator_id == "Victor"]["case_number"].tolist()

parker1, parker2 = split_half(parker_cases)
brian1,  brian2  = split_half(brian_cases)
victor1, victor2 = split_half(victor_cases)

# ───────────────────────────────────────────
# 4. Create ROUND 2 assignments
# ───────────────────────────────────────────
round2 = []

# Parker → Brian + Victor
round2 += [{"case_number": c, "annotator_id": "Brian"} for c in parker1]
round2 += [{"case_number": c, "annotator_id": "Victor"} for c in parker2]

# Brian → Parker + Victor
round2 += [{"case_number": c, "annotator_id": "Parker"} for c in brian1]
round2 += [{"case_number": c, "annotator_id": "Victor"} for c in brian2]

# Victor → Brian + Parker
round2 += [{"case_number": c, "annotator_id": "Brian"} for c in victor1]
round2 += [{"case_number": c, "annotator_id": "Parker"} for c in victor2]

round2_df = pd.DataFrame(round2)

# ───────────────────────────────────────────
# 5. VERIFICATIONS BEFORE INSERTING
# ───────────────────────────────────────────
print("\n🔍 Verifying…")

n1 = df["case_number"].nunique()
n2 = round2_df["case_number"].nunique()
print("Round1 unique (3 annotators only):", n1)
print("Round2 unique:", n2)

assert n1 == n2, "❌ Round 2 does NOT contain the same unique cases!"

# No duplicates
assert round2_df["case_number"].duplicated().sum() == 0, "❌ Duplicate cases in round 2!"

# No self-assign
merged = round2_df.merge(df[["case_number", "annotator_id"]], on="case_number", suffixes=("_r2", "_r1"))
assert (merged.annotator_id_r1 == merged.annotator_id_r2).sum() == 0, "❌ Self-assignment detected!"

# Distribution OK
print(round2_df["annotator_id"].value_counts())

# Set equality
assert set(df["case_number"]) == set(round2_df["case_number"]), "❌ Case sets differ!"

# ───────────────────────────────────────────
# 6. Add metadata + insert
# ───────────────────────────────────────────
final = round2_df.merge(
    df[["case_number", "plaintiff", "defendant", "complaint_filed_date", "case_hashkey"]],
    on="case_number",
    how="left"
)

final["round"] = 2
final["progress"] = "incomplete"

records = final.to_dict(orient="records")

print("\n🚀 Inserting round 2 into Supabase…")
supabase.table("cases_gold").insert(records).execute()

print("✅ ROUND 2 SUCCESSFULLY INSERTED!")
print("Inserted rows:", len(records))
print(final["annotator_id"].value_counts())


Loaded round 1 cases total: 111
Cases retained for IAA: 105
annotator_id
Parker    35
Victor    35
Brian     35
Name: count, dtype: int64

🔍 Verifying…
Round1 unique (3 annotators only): 105
Round2 unique: 105
annotator_id
Victor    36
Parker    35
Brian     34
Name: count, dtype: int64

🚀 Inserting round 2 into Supabase…
✅ ROUND 2 SUCCESSFULLY INSERTED!
Inserted rows: 105
annotator_id
Victor    36
Parker    35
Brian     34
Name: count, dtype: int64


In [7]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ──────────────────────────────────────────────
# 1. Load Supabase
# ──────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

ANNOTATORS = ["Parker", "Brian", "Victor"]
IGNORE = {"time_caselevel", "created_at"}

# ──────────────────────────────────────────────
# Function to compare for one annotator
# ──────────────────────────────────────────────
def compare_for(annotator):
    print("\n" + "="*60)
    print(f"🔎 Checking annotator: {annotator}")
    print("="*60)

    # Fetch gold
    df_gold = pd.DataFrame(
        supabase.table("results_gold")
        .select("*")
        .eq("annotator_id", annotator)
        .execute().data
    )

    # Fetch fallback
    df_fb = pd.DataFrame(
        supabase.table("results_gold_fallback")
        .select("*")
        .eq("annotator_id", annotator)
        .execute().data
    )

    print(f"rows in results_gold: {len(df_gold)}")
    print(f"rows in fallback: {len(df_fb)}")

    if df_gold.empty or df_fb.empty:
        print(f"❌ No data for annotator {annotator}")
        return

    # Keep last version per case
    last_gold = (
        df_gold.sort_values(["case_number", "version"])
        .groupby("case_number")
        .tail(1)
        .reset_index(drop=True)
    )
    last_fb = (
        df_fb.sort_values(["case_number", "version"])
        .groupby("case_number")
        .tail(1)
        .reset_index(drop=True)
    )

    cases_gold = set(last_gold.case_number)
    cases_fb = set(last_fb.case_number)

    common = sorted(cases_gold & cases_fb)

    # Warn if mismatch
    if cases_gold != cases_fb:
        print("⚠️ Case mismatch:")
        print("  In gold only:", cases_gold - cases_fb)
        print("  In fallback only:", cases_fb - cases_gold)

    # Compare ignoring IGNORE fields
    diffs = []
    for case in common:
        g = last_gold[last_gold.case_number == case].iloc[0].to_dict()
        f = last_fb[last_fb.case_number == case].iloc[0].to_dict()

        cols = set(g.keys()) | set(f.keys())
        for col in cols:
            if col in IGNORE:
                continue

            gv = g.get(col)
            fv = f.get(col)

            if isinstance(gv, list): gv = sorted(gv)
            if isinstance(fv, list): fv = sorted(fv)

            if pd.isna(gv) and pd.isna(fv):
                continue

            if gv != fv:
                diffs.append({
                    "case_number": case,
                    "column": col,
                    "gold": gv,
                    "fallback": fv
                })

    if not diffs:
        print(f"✅ PERFECT MATCH for {annotator} (ignoring time_caselevel + created_at)")
    else:
        print(f"❌ DIFFERENCES FOUND for {annotator}:")
        print(pd.DataFrame(diffs).to_string(index=False))


# ──────────────────────────────────────────────
# Run comparison for all three
# ──────────────────────────────────────────────
for annotator in ANNOTATORS:
    compare_for(annotator)





🔎 Checking annotator: Parker
rows in results_gold: 108
rows in fallback: 71
⚠️ Case mismatch:
  In gold only: {'23CHLC16737'}
  In fallback only: set()
✅ PERFECT MATCH for Parker (ignoring time_caselevel + created_at)

🔎 Checking annotator: Brian
rows in results_gold: 125
rows in fallback: 90
✅ PERFECT MATCH for Brian (ignoring time_caselevel + created_at)

🔎 Checking annotator: Victor
rows in results_gold: 95
rows in fallback: 60
✅ PERFECT MATCH for Victor (ignoring time_caselevel + created_at)


# Upload second batch

In [2]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load data from the specified CSV
# ────────────────────────────────────────────────
data = pd.read_csv("/Users/othmanbensouda/Desktop/debt_collection_website/files/get_plaintiffs_defendants_initial_100.csv")

# ────────────────────────────────────────────────
# 3. Clean up text fields (avoid NaN/float errors)
# ────────────────────────────────────────────────
data["FULL_NAME"] = data["FULL_NAME"].fillna("").astype(str)
data["PERSON_ROLE"] = data["PERSON_ROLE"].fillna("").astype(str)

# ────────────────────────────────────────────────
# 4. Aggregate one row per case
# ────────────────────────────────────────────────
grouped = (
    data.groupby("CASE_NUMBER")
    .apply(lambda g: pd.Series({
        "case_number": g.name,
        "case_hashkey": g["CASE_HASHKEY"].iloc[0] if "CASE_HASHKEY" in g else None,
        "complaint_filed_date": g["COMPLAINT_FILED_DATE"].iloc[0] if "COMPLAINT_FILED_DATE" in g else None,
        "plaintiff": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "plaintiff", "FULL_NAME"] if pd.notna(x) and x],
        "defendant": [str(x) for x in g.loc[g["PERSON_ROLE"].str.lower() == "defendant", "FULL_NAME"] if pd.notna(x) and x],
        "annotator_id": None,  # No annotator assigned for batch 2
        "batch": 2  # Set batch to 2
    }))
    .reset_index(drop=True)
)

# ────────────────────────────────────────────────
# 5. Upload to Supabase (batch, fast)
# ────────────────────────────────────────────────
records = grouped.to_dict(orient="records")

response = supabase.table("cases_gold").upsert(records).execute()

print("✅ Upload complete!")
print("Inserted/updated rows:", len(records))
print("All rows set to batch = 2")

# Optional: preview one example
print("\nExample row:")
print(grouped.head(1).to_dict(orient="records")[0])

/var/folders/k7/b0_b7t6j6n72t68sh4s7t8400000gn/T/ipykernel_21564/1567906470.py:30: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


✅ Upload complete!
Inserted/updated rows: 99
All rows set to batch = 2

Example row:
{'case_number': '23CHLC04088', 'case_hashkey': 938445792436271488, 'complaint_filed_date': '2023-02-14 00:00:00.000', 'plaintiff': ['INVESTMENT RETRIEVERS, INC.'], 'defendant': ['KANDIS MCDONALD'], 'annotator_id': None, 'batch': 2}


# Send initial 100 debt collection cases google drive links to supabase

In [3]:
import os
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Load data from CSV
# ────────────────────────────────────────────────
data = pd.read_csv("/Users/othmanbensouda/Desktop/debt_collection_website/files/initial_100_debt_collection_gold_list.csv")

print("Column names in CSV:")
print(data.columns.tolist())
print("\nFirst few rows:")
print(data.head())

# ────────────────────────────────────────────────
# 3. Clean up and prepare data
# ────────────────────────────────────────────────
# Fill NaN values with empty strings to avoid issues
data = data.fillna("")

# Convert to records for upload
records = data.to_dict(orient="records")

# ────────────────────────────────────────────────
# 4. Upload to Supabase gdrive_files table
# ────────────────────────────────────────────────
response = supabase.table("gdrive_files").upsert(records).execute()

print("\n✅ Upload complete!")
print("Inserted/updated rows:", len(records))

Column names in CSV:
['CASE_NUMBER', 'DOCUMENT_ID', 'DOCUMENT_FILED_DATE', 'DOCUMENT_NAME', 'DOCUMENT_NAME_DRIVE', 'LINK_DRIVE']

First few rows:
   CASE_NUMBER  DOCUMENT_ID      DOCUMENT_FILED_DATE  \
0  23CHLC04088     91598076  2023-02-14 00:00:00.000   
1  23CHLC04088     91598077  2023-02-14 00:00:00.000   
2  23CHLC04088     91598078  2023-02-14 00:00:00.000   
3  23CHLC04088     91598079  2023-02-14 00:00:00.000   
4  23CHLC04088     91598080  2023-02-14 00:00:00.000   

                                       DOCUMENT_NAME  \
0                                          Complaint   
1                                            Summons   
2                       Declaration (name extension)   
3                             Civil Case Cover Sheet   
4  Order to Show Cause Hearing/Case Management Re...   

                DOCUMENT_NAME_DRIVE  \
0    complaint_23chlc04088_91598076   
1                               NaN   
2  declaration_23chlc04088_91598078   
3                       

# Assign the initial 100 cases randomly

In [6]:
import os
import random
import pandas as pd
from supabase import create_client
from dotenv import load_dotenv

# ────────────────────────────────────────────────
# 1. Load Supabase credentials
# ────────────────────────────────────────────────
load_dotenv()
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ────────────────────────────────────────────────
# 2. Fetch all batch 2 cases from cases_gold
# ────────────────────────────────────────────────
response = supabase.table("cases_gold").select("*").eq("batch", 2).execute()
batch_2_cases = response.data

if not batch_2_cases:
    print("⚠️ No cases found in batch 2")
    exit()

print(f"Found {len(batch_2_cases)} cases in batch 2")

# ────────────────────────────────────────────────
# 3. Assign annotators randomly (Brian, Parker, Victor)
# ────────────────────────────────────────────────
case_numbers = [case["case_number"] for case in batch_2_cases]
random.seed(42)
random.shuffle(case_numbers)
n = len(case_numbers)

part1 = case_numbers[: n // 3]
part2 = case_numbers[n // 3 : 2 * n // 3]
part3 = case_numbers[2 * n // 3 :]

def assign_annotator(case_number):
    if case_number in part1:
        return "Brian"
    elif case_number in part2:
        return "Parker"
    else:
        return "Victor"

# ────────────────────────────────────────────────
# 4. Update each case with annotator_id
# ────────────────────────────────────────────────
updates = []
successful_updates = 0
failed_updates = 0

for case in batch_2_cases:
    annotator = assign_annotator(case["case_number"])
    
    # Update in Supabase
    try:
        response = supabase.table("cases_gold")\
            .update({"annotator_id": annotator})\
            .eq("case_number", case["case_number"])\
            .eq("batch", 2)\
            .execute()
        successful_updates += 1
    except Exception as e:
        print(f"Error updating case {case['case_number']}: {e}")
        failed_updates += 1
    
    # Store for Excel export
    updates.append({
        "case_number": case["case_number"],
        "batch": 2,
        "case_hashkey": case.get("case_hashkey"),
        "complaint_filed_date": case.get("complaint_filed_date"),
        "plaintiff": case.get("plaintiff"),
        "defendant": case.get("defendant"),
        "annotator_id": annotator
    })

print("✅ Assignment complete!")
print(f"Successfully updated: {successful_updates}")
print(f"Failed updates: {failed_updates}")
print(f"Total cases: {len(batch_2_cases)}")

# ────────────────────────────────────────────────
# 5. Create DataFrame and save to Excel
# ────────────────────────────────────────────────
df = pd.DataFrame(updates)

# Sort by annotator for easier review
df = df.sort_values(["annotator_id", "case_number"]).reset_index(drop=True)

# Save to Excel
output_path = "/Users/othmanbensouda/Desktop/debt_collection_website/files/batch_2_case_assignments.xlsx"
df.to_excel(output_path, index=False, engine='openpyxl')

print(f"\n✅ Excel file saved to: {output_path}")

# Show distribution
print("\nAnnotator distribution:")
print(df["annotator_id"].value_counts().sort_index())

# Show sample of each annotator's assignments
print("\nSample assignments per annotator:")
for annotator in ["Brian", "Parker", "Victor"]:
    print(f"\n{annotator}:")
    annotated_cases = df[df["annotator_id"] == annotator]["case_number"].head(3).tolist()
    print(annotated_cases)

Found 99 cases in batch 2
✅ Assignment complete!
Successfully updated: 99
Failed updates: 0
Total cases: 99

✅ Excel file saved to: /Users/othmanbensouda/Desktop/debt_collection_website/files/batch_2_case_assignments.xlsx

Annotator distribution:
annotator_id
Brian     33
Parker    33
Victor    33
Name: count, dtype: int64

Sample assignments per annotator:

Brian:
['23CHLC12580', '23NWLC25562', '23NWLC29036']

Parker:
['23CHLC12118', '23CHLC16737', '23CHLC22869']

Victor:
['23CHLC04088', '23CHLC18504', '23CHLC18998']
